In [ ]:
import xgboost as xgb
import pandas as pd
import src.train_utils as T
from sklearn.dummy import DummyRegressor

In [4]:
df = pd.read_csv('../datasets/base.csv')
df['date'] = pd.to_datetime(df['date'])

train_df = df[df['date'].dt.year < 2022]

In [52]:
frp_exp_df = T.add_spatial_lags(train_df, [3, 5, 7], ['frp_t'], pool='mean')
frp_exp_df = T.add_spatial_lags(frp_exp_df, [3, 5, 7], ['frp_t'], pool='max')

In [53]:
frp_exp_df = frp_exp_df.dropna()

In [54]:
base = []

feature_experiments = [
    ('persistence', base),
]

model=DummyRegressor(strategy='constant', constant=0)

results = T.run_experiments(
    df=frp_exp_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: persistence
Fold 0
RMSE: 1.7746019244355444
Fold 1
RMSE: 6.269512050039968
Fold 2
RMSE: 4.223291823935145
Fold 3
RMSE: 8.669010890470068
Fold 4
RMSE: 14.754026021929128
Fold 5
RMSE: 4.4301957827731835
Fold 6
RMSE: 0.9896191160986055
Fold 7
RMSE: 1.0655872014374173
Fold 8
RMSE: 1.653758701602382
Fold 9
RMSE: 4.126343254845557
    experiment  n_features  unweighted_rmse  weighted_rmse
0  persistence           0         4.795595        6.63755


In [58]:
base = ['pm25_t', 'u_wind_10m_t', 'v_wind_10m_t', 'delta_pm25_t', 'precip_sum_t', 'dew_temp_2m_t', 'temp_2m_t', 'frp_t']

params = {
    'max_depth': 3,            
    'learning_rate': 0.1,     
    'n_estimators': 45,     
    'subsample': 0.8,          
    'colsample_bytree': 0.8,   
}

feature_experiments = [
    # Control
    ('base', base),
    # Mean FRP
    ('base + 3x3 FRP mean', base + ['frp_t_mean_3x3']),
    ('base + 5x5 FRP mean', base + ['frp_t_mean_5x5']),
    ('base + 7x7 FRP mean', base + ['frp_t_mean_7x7']),
    ('base + 3x3 FRP + 5x5 FRP mean', base + ['frp_t_mean_3x3'] + ['frp_t_mean_5x5']),
    ('base + 3x3 FRP + 5x5 FRP + 7x7 FRP mean', base + ['frp_t_mean_3x3'] + ['frp_t_mean_5x5'] + ['frp_t_mean_7x7']),
    # Max FRP
    ('base + 3x3 FRP max', base + ['frp_t_max_3x3']),
    ('base + 5x5 FRP max', base + ['frp_t_max_5x5']),
    ('base + 7x7 FRP max', base + ['frp_t_max_7x7']),
    ('base + 3x3 FRP + 5x5 FRP max', base + ['frp_t_max_3x3'] + ['frp_t_max_5x5']),
    ('base + 3x3 FRP + 5x5 FRP + 7x7 FRP max', base + ['frp_t_max_3x3'] + ['frp_t_max_5x5'] + ['frp_t_max_7x7'])
]

model=xgb.XGBRegressor(**params, random_state=191)

results = T.run_experiments(
    df=frp_exp_df, 
    model=model, 
    feature_experiments=feature_experiments, 
    train_days=365*2,
    gap_days=21,
    val_days=49
)

print(results.sort_values('weighted_rmse'))

Running experiment: base
Fold 0
RMSE: 1.7430977950772895
Fold 1
RMSE: 5.548818489410973
Fold 2
RMSE: 4.291156847487845
Fold 3
RMSE: 8.305124691155967
Fold 4
RMSE: 13.518189883676246
Fold 5
RMSE: 4.137517347874542
Fold 6
RMSE: 0.9891661690741027
Fold 7
RMSE: 1.0779038847734281
Fold 8
RMSE: 1.6550192327590059
Fold 9
RMSE: 4.044288966208013
Running experiment: base + 3x3 FRP mean
Fold 0
RMSE: 1.746659015676512
Fold 1
RMSE: 5.522572691571345
Fold 2
RMSE: 4.307426778042061
Fold 3
RMSE: 8.318495093476809
Fold 4
RMSE: 13.405678789323211
Fold 5
RMSE: 4.01458248395524
Fold 6
RMSE: 0.9834578954668657
Fold 7
RMSE: 1.07755602879506
Fold 8
RMSE: 1.6521578146553408
Fold 9
RMSE: 4.068367717436219
Running experiment: base + 5x5 FRP mean
Fold 0
RMSE: 1.738724242246977
Fold 1
RMSE: 5.469638753130348
Fold 2
RMSE: 4.279311692691982
Fold 3
RMSE: 8.280723497826033
Fold 4
RMSE: 13.303442972338965
Fold 5
RMSE: 4.060881900670092
Fold 6
RMSE: 0.9860895990122989
Fold 7
RMSE: 1.0775812435899268
Fold 8
RMSE: 1.642